# Занятие 4. Геометрия на плоскости

**Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026**
Лекция-опора: **L4 «Геометрические преобразования и ресемплинг»** · **90 минут в классе** (первые 15 — Quiz 1) · ведёт ассистент

---

Сегодня собираем **«выпрямитель»**: четыре угла плаката в кадре → порядок tl, tr, br, bl → гомография
`3 × 3` → фронтальный вид с честными пропорциями — и измеряем, чего стоит ошибка клика в пикселях.
Та же матрица переводит кадр с камеры робота в **вид сверху в метрах** (bird's-eye view). Во второй
половине меряем **четыре интерполяции** числом при увеличении ×8 и находим, где флаг уже не помогает.

Заготовка начинается ровно на демо 1.2, 2.2 и 2.3 лекции: вы это уже видели, теперь — руками.

| Минуты | Что происходит |
|--------|----------------|
| 0–15 | **Quiz 1** (блок I, L1–L3) в Moodle — если проводится на паре |
| 15–25 | Ассистент разбирает опорный пример: четыре точки → фронтальный вид; что гомография сохраняет, а что нет |
| 25–65 | Вы делаете **TODO 1–3** |
| 65–75 | Разбор решения |
| 75–85 | **Мост к ДЗ 1 «Фотолаборатория»**: пункт «геометрия» |
| 85–90 | Зачёт |

Ничего сдавать не нужно: **зачёт ставится в классе** по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется в репозитории сразу после занятия.

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"     # адрес репозитория курса (для Colab)

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:      # Colab: пакет курса ставится один раз за сессию
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt   (docs/setup-guide.md)")

import os, time, timeit, itertools
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём данные

Всё лежит в репозитории, в папке лекции `lectures/L04-geometry/demo/` — скачивать нечего:

- `poster-angle.jpg` — фото здания ИУ, на фасаде «висит» плакат под углом. **Его углы в кадре известны
  точно:** (560, 330), (1250, 220), (1290, 1010), (600, 840) — это эталон, по нему считаем ошибки в пикселях;
- `poster-gt.png` — тот же плакат «в лоб», 640 × 905 (≈ A4): с ним сравниваем выпрямленный вид;
- `road-cam.jpg` — кадр с камеры робота (модель камеры из L2: fx 1663, высота 1.2 м, наклон 12°) для bird's-eye view;
- `text-page.png` и `iu-building.jpg` — страница текста и фото для сравнения интерполяций.

Источники по порядку: репозиторий рядом → скачать с GitHub (Colab) → синтетическая сцена (работает всегда,
фон другой — числа другие). **Своё фото** плаката, этикетки или документа под углом — в `data/my-poster.jpg`,
к нему вернёмся после TODO 2.

In [ ]:
LECTURE_DIR = "lectures/L04-geometry/demo"
CANDIDATE_DIRS = ["../../" + LECTURE_DIR,                     # публичный репозиторий: labs/lab04-geometry/
                  "../../../iu-intro-cv/" + LECTURE_DIR,      # ноутбук открыт из iu-intro-cv-ta
                  "data"]                                     # уже скачанное
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
FILES = ("poster-angle.jpg", "poster-gt.png", "road-cam.jpg", "text-page.png", "iu-building.jpg")
POSTER = np.float32([[560, 330], [1250, 220], [1290, 1010], [600, 840]])   # углы плаката в кадре: tl, tr, br, bl (эталон)
POSTER_SIZE = (640, 905)                                                   # плакат «в лоб»: (ширина, высота), ≈ A4


def draw_poster(w=640, h=905):
    """Плакат «в лоб» — рисунок лекции: рамка, заголовок, «логотип», сетка 4 × 4, круг, линейка."""
    p = np.full((h, w, 3), 255, np.uint8)
    cv2.rectangle(p, (6, 6), (w - 7, h - 7), (90, 90, 90), 2)
    cv2.putText(p, "Intro to Computer Vision", (36, 70), cv2.FONT_HERSHEY_DUPLEX, 1.15, (30, 30, 30), 2, cv2.LINE_AA)
    cv2.rectangle(p, (36, 150), (216, 250), (192, 112, 0), -1)
    cv2.putText(p, "IU", (78, 224), cv2.FONT_HERSHEY_DUPLEX, 2.0, (255, 255, 255), 3, cv2.LINE_AA)
    x0, y0, cell = 60, 320, 130
    for i in range(5):
        cv2.line(p, (x0, y0 + i * cell), (x0 + 4 * cell, y0 + i * cell), (0, 0, 0), 2)
        cv2.line(p, (x0 + i * cell, y0), (x0 + i * cell, y0 + 4 * cell), (0, 0, 0), 2)
    cv2.circle(p, (x0 + 195, y0 + 195), 52, (0, 0, 200), 3, cv2.LINE_AA)
    cv2.line(p, (20, 870), (620, 870), (0, 0, 0), 2)
    for x in range(0, 601, 50):
        cv2.line(p, (20 + x, 870), (20 + x, 870 - (18 if x % 100 == 0 else 9)), (0, 0, 0), 2)
    return p


def synthetic_scenes():
    """Запас без сети: плакат на сером градиенте по тем же углам; дорога — разметка по той же модели камеры."""
    gt = draw_poster()
    bg = np.tile(np.linspace(150, 90, 1920).astype(np.uint8)[None, :, None], (1279, 1, 3))
    Hp = cv2.getPerspectiveTransform(np.float32([[0, 0], [639, 0], [639, 904], [0, 904]]), POSTER)
    m = cv2.warpPerspective(np.full(gt.shape[:2], 255, np.uint8), Hp, (1920, 1279)) > 127
    scene = bg.copy(); scene[m] = cv2.warpPerspective(gt, Hp, (1920, 1279))[m]
    th = np.deg2rad(12.0); K = np.array([[1663, 0, 960], [0, 1663, 540], [0, 0, 1]], float)
    Hgi = K @ np.array([[1, 0, 0], [0, -np.sin(th), 1.2 * np.cos(th)], [0, np.cos(th), 1.2 * np.sin(th)]])
    tex = np.full((1600, 480, 3), 96, np.uint8)                       # вид сверху 40 px/м: X −6…6 м, Z 0…40 м
    XX, ZZ = np.meshgrid(-6 + (np.arange(480) + 0.5) / 40, 40 - (np.arange(1600) + 0.5) / 40)
    tex[np.abs(XX) > 3.7] = (70, 125, 70)
    tex[(np.abs(np.abs(XX) - 3.55) < 0.07) | (np.abs(np.abs(XX) - 1.75) < 0.07) | ((np.abs(XX) < 0.06) & (ZZ % 3 < 1.5))] = 235
    road = cv2.warpPerspective(tex, Hgi @ np.array([[1 / 40, 0, -6], [0, -1 / 40, 40], [0, 0, 1]]), (1920, 1080), borderValue=(200, 170, 120))
    text = np.full((480, 640), 255, np.uint8)
    for i, t in enumerate(("Intro to Computer Vision", "Lecture 4: geometry", "inverse warping", "x' ~ H x")):
        cv2.putText(text, t, (24, 80 + 100 * i), cv2.FONT_HERSHEY_SIMPLEX, 1.0, 0, 2, cv2.LINE_AA)
    return scene, gt, road, text, scene.copy()


def load_data():
    read = lambda d: [cio.imread(os.path.join(d, f), "gray" if f == "text-page.png" else "color") for f in FILES]
    for d in CANDIDATE_DIRS:
        if all(os.path.exists(os.path.join(d, f)) for f in FILES):
            return read(d) + [d]
    try:                                                   # Colab: репозитория рядом нет — качаем
        import urllib.request
        os.makedirs("data", exist_ok=True)
        for f in FILES:
            urllib.request.urlretrieve(RAW_URL + f, os.path.join("data", f))
        return read("data") + [RAW_URL]
    except Exception as e:
        print("файлы лекции не скачались (%s) — синтетическая сцена" % type(e).__name__)
        return list(synthetic_scenes()) + ["синтетика"]


scene, poster_gt, road, text_page, photo, SRC = load_data()
assert scene.shape[:2] == (1279, 1920) and poster_gt.shape[:2] == (905, 640) and road.shape[:2] == (1080, 1920)
print("источник:", SRC, "· кадр", scene.shape, "· плакат «в лоб»", poster_gt.shape, "· дорога", road.shape, "· текст", text_page.shape)
shown = scene.copy(); cv2.polylines(shown, [np.int32(POSTER)], True, viz.ORANGE, 5)
viz.grid({"кадр и четыре угла плаката": shown, "плакат «в лоб» (эталон)": poster_gt, "камера робота": road}, cols=3, size=3.6)

## 2. Опорный пример — разбирает ассистент

Это демо 1.2 лекции и слайд 18 («Пример на числах»). Здесь всё написано, задача — **понять каждую строку**.

**Четыре пары точек → одна матрица.** Углы плаката в кадре `src` и углы листа `dst` — **в одном и том же
порядке** tl, tr, br, bl. `cv2.getPerspectiveTransform(src, dst)` решает систему из 8 уравнений точно:
четыре опорные точки воспроизводятся до 10⁻⁶, ошибка живёт только в *других* точках.
`cv2.warpPerspective(img, H, dsize)` исполняет её обратным варпом; `dsize` — **(ширина, высота)**.

**Метрика — геометрическая.** Яркость фасада и плаката «в лоб» разная, PSNR это считает ошибкой; поэтому
обе картинки бинаризуем (порог 128, как в L3) и считаем **долю совпавших пикселей**: слайд 5 — 99.1 %.

In [ ]:
def binary_match(a, b):
    """Геометрия, а не яркость: обе картинки → «чернила» (серый < 128) → доля совпавших пикселей, %."""
    ink = lambda im: (cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if im.ndim == 3 else im) < 128
    return 100.0 * float((ink(a) == ink(b)).mean())


def apply_H(H, pts):
    """Применить 3 × 3 к точкам (N, 2): однородные координаты и деление на w."""
    p = np.c_[np.asarray(pts, np.float64).reshape(-1, 2), np.ones(len(np.reshape(pts, (-1, 2))))] @ H.T
    return p[:, :2] / p[:, 2:3]


W, Hh = POSTER_SIZE
dst = np.float32([[0, 0], [W - 1, 0], [W - 1, Hh - 1], [0, Hh - 1]])   # углы листа — в том же порядке tl, tr, br, bl
H_ref = cv2.getPerspectiveTransform(POSTER, dst)                        # 3 × 3, кадр → лист; точное решение
front_ref = cv2.warpPerspective(scene, H_ref, (W, Hh), flags=cv2.INTER_LINEAR)   # dsize = (ширина, высота)!

side = lambda a, b: float(np.linalg.norm(POSTER[a] - POSTER[b]))
print("стороны плаката в кадре: верх %.0f, низ %.0f, лево %.0f, право %.0f px — правая на %.0f %% длиннее левой"
      % (side(0, 1), side(3, 2), side(0, 3), side(1, 2), 100 * (side(1, 2) / side(0, 3) - 1)))
print("пропорции W/H «по кадру» %.2f, на самом деле %.3f (A4): ни поворот, ни resize этого не исправят"
      % (0.5 * (side(0, 1) + side(3, 2)) / (0.5 * (side(0, 3) + side(1, 2))), W / Hh))
print("H =", np.round(H_ref, 4).tolist())
print("углы кадра → лист:", np.round(apply_H(H_ref, POSTER), 3).tolist())
print("выпрямленный вид против плаката «в лоб»: совпало %.1f %% пикселей (PSNR был бы %.1f дБ — яркость фасада другая)"
      % (binary_match(front_ref, poster_gt), metrics.psnr(poster_gt, front_ref)))
viz.grid({"кадр": shown, "выпрямленный вид 640 × 905": front_ref, "эталон «в лоб»": poster_gt}, cols=3, size=3.6)

### Что гомография сохраняет, а что нет

Демо 1.3 и слайд 13. На линейке плаката (нижний край, деления через 50 px) берём четыре точки A, B, C, D
и переносим их в кадр через `H⁻¹`. **Отношение длин** AB/BC на листе 0.5 — в кадре другое: гомография его
не сохраняет. **Двойное отношение** (AC·BD)/(BC·AD) — одинаковое: это инвариант проективного преобразования.
А параллельные горизонтали сетки в кадре **сходятся** — в точку схода, образ точки на бесконечности.

In [ ]:
A, B, C, D = pts_sheet = np.float64([[20, 870], [120, 870], [320, 870], [620, 870]])   # деления линейки: 0, 100, 300, 600
pts_frame = apply_H(np.linalg.inv(H_ref), pts_sheet)                                   # те же точки в кадре
d = lambda p, q: float(np.linalg.norm(p - q))
cross = lambda u: d(u[0], u[2]) * d(u[1], u[3]) / (d(u[1], u[2]) * d(u[0], u[3]))
print("AB/BC:              лист %.3f, кадр %.3f   ← простое отношение не сохраняется"
      % (d(A, B) / d(B, C), d(pts_frame[0], pts_frame[1]) / d(pts_frame[1], pts_frame[2])))
print("двойное отношение:  лист %.4f, кадр %.4f   ← инвариант гомографии" % (cross(pts_sheet), cross(pts_frame)))
vp = np.linalg.inv(H_ref) @ np.array([1.0, 0.0, 0.0])                                   # направление «вправо» на листе: точка с w = 0
print("горизонтали листа сходятся в кадре в точке схода (%.1f, %.1f) — за левым краем кадра" % (vp[0] / vp[2], vp[1] / vp[2]))

### Вопрос, на который отвечаем вслух

Правая сторона плаката в кадре на 55 % длиннее левой. Какой ступени иерархии (сдвиг → евклидово → подобие →
аффинное → проективное) **хватит**, чтобы это исправить, и почему `resize` и `getRotationMatrix2D` бессильны?
Сколько пар точек нужно этой ступени?

---

## TODO 1 — порядок углов *(≈ 8 минут)*

Кликов по углам будет четыре, но **в каком порядке** — как повезёт. А `getPerspectiveTransform` требует,
чтобы `src` и `dst` шли попарно: перепутали две точки — лист сложится «бабочкой» **без единого
предупреждения** (слайд 69).

Напишите `order_corners(pts)`: четыре точки в любом порядке → `np.float32` формы `(4, 2)` в порядке
**tl, tr, br, bl**. Правило со слайда лекции (рецепт курса): у **tl** минимальна сумма `x + y`, у **br** —
максимальна; у **tr** минимальна разность `y − x`, у **bl** — максимальна.

> **Подсказка.** `p.sum(1)` — суммы по строкам; `np.argmin`, `np.argmax` — индексы; результат собрать
> из `p[i]` и привести к `np.float32`.

In [ ]:
def order_corners(pts):
    """Четыре точки (в любом порядке) → float32 (4, 2) в порядке tl, tr, br, bl."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 4–6 строк: суммы x + y (tl — минимум, br — максимум) и разности y − x (tr — минимум, bl — максимум)
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────


for perm in itertools.permutations(range(4)):
    got = order_corners(POSTER[list(perm)])
    assert got.shape == (4, 2) and got.dtype == np.float32, "ожидался np.float32 формы (4, 2)"
    assert np.array_equal(got, POSTER), "порядок %s должен превращаться в tl, tr, br, bl (сейчас %s)" % (perm, got.tolist())
box = order_corners([(300, 50), (100, 50), (100, 200), (300, 200)])
assert np.array_equal(box, [[100, 50], [300, 50], [300, 200], [100, 200]]), "прямоугольник: tl, tr, br, bl"
print("TODO 1 ✔  · все 24 порядка углов плаката приводятся к tl, tr, br, bl")

### Где правило ломается — проверяем измерением

Правило «суммы и разности» есть в каждом туториале. Возьмём лист A4 (те же 640 × 905, вдвое меньше),
повернём его в кадре на угол θ от 0 до 175°, перемешаем углы и посмотрим, при каких θ `order_corners`
возвращает **не циклический сдвиг** истинного порядка (или две одинаковые точки — тогда H вырождена).

In [ ]:
sheet = np.float32([[0, 0], [319, 0], [319, 452], [0, 452]]) - [160, 226]                   # лист вдвое меньше, центр в нуле
rot = lambda deg: (sheet @ cv2.getRotationMatrix2D((0, 0), deg, 1.0)[:, :2].T + [960, 640]).astype(np.float32)
rng = np.random.default_rng(4)
broken = []
for deg in range(0, 180, 5):
    true = rot(deg)                                                                          # tl, tr, br, bl после поворота
    got = order_corners(true[rng.permutation(4)])
    cyclic = any(np.allclose(got, np.roll(true, -k, axis=0)) for k in range(4))
    if not cyclic:
        broken.append(deg)
print("правило x + y / y − x даёт не циклический порядок при θ =", broken, "→ H по таким точкам — «бабочка» или вырождение")


def order_by_angle(pts):
    """Порядок по углу вокруг центроида — для выпуклого четырёхугольника цикл всегда; первым — угол с минимальной x + y."""
    p = np.asarray(pts, np.float32).reshape(4, 2)
    p = p[np.argsort(np.arctan2(*(p - p.mean(0)).T[::-1]))]                                  # по часовой (ось y вниз)
    return np.roll(p, -int(np.argmin(p.sum(1))), axis=0)


broken2 = [deg for deg in range(0, 180, 5)
           if not any(np.allclose(order_by_angle(rot(deg)[rng.permutation(4)]), np.roll(rot(deg), -k, axis=0)) for k in range(4))]
print("порядок по углу вокруг центроида: не циклический при θ =", broken2 or "— никогда")

### Вопрос, на который отвечаем вслух

Для документа, снятого «примерно ровно», правило сумм и разностей работает; для листа, лежащего на столе
под углом 45°, — нет. Почему `getPerspectiveTransform` с перепутанными точками **не бросает исключение**,
и по каким числам «бабочку» можно заметить до того, как смотреть на картинку? (Подсказка: слайд 57 —
`|det H|`, углы результата через `perspectiveTransform`.)

---

## TODO 2 — выпрямитель *(≈ 15 минут)*

Напишите `rectify(img, corners, size=None, aspect=None)` → `(front, H)`:

1. углы — через `order_corners` (порядок на входе любой);
2. размер выхода `(W, H)`: если задан `size` — он; иначе `W` = длина большей из горизонтальных сторон
   в кадре (top / bottom), а `H` = `W / aspect`, если известны **реальные пропорции** объекта
   (A4 — 210/297), иначе — большая из вертикальных сторон в кадре;
3. `dst` — углы листа `(0, 0), (W−1, 0), (W−1, H−1), (0, H−1)`;
4. `H = cv2.getPerspectiveTransform(src, dst)`;
5. `front = cv2.warpPerspective(img, H, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)`.

Это рецепт курса (шаги 2–4) — он же пойдёт в ДЗ 1 и в ДЗ 2 «Сканер документов».

> **Подсказка.** `np.linalg.norm(c[1] - c[0])` — длина стороны tl–tr. `dsize` в `warpPerspective` —
> кортеж **(ширина, высота)** из целых; `img.shape[:2]` — это наоборот.

In [ ]:
def rectify(img, corners, size=None, aspect=None):
    """Четыре угла (любой порядок) → (фронтальный вид, H кадр → лист). size = (W, H); aspect = W / H, если size не задан."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 8–11 строк: order_corners; размер из size / aspect / длин сторон; dst-углы; getPerspectiveTransform; warpPerspective с dsize = (W, H)
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


front, H = rectify(scene, POSTER[[2, 0, 3, 1]], size=POSTER_SIZE)               # углы — в перепутанном порядке
assert front.shape[:2] == (POSTER_SIZE[1], POSTER_SIZE[0]), "dsize = (W, H): результат должен быть 905 строк × 640 столбцов"
assert np.abs(apply_H(H, POSTER) - dst).max() < 1e-3, "H должна переводить углы кадра ровно в углы листа (проверьте порядок точек)"
match = binary_match(front, poster_gt)
assert match > 98.5, "выпрямленный вид должен совпасть с плакатом «в лоб» на > 98.5 %% (сейчас %.1f)" % match
assert np.array_equal(front, rectify(scene, POSTER, size=POSTER_SIZE)[0]), "порядок точек на входе не должен влиять на результат"
f_a4, _ = rectify(scene, POSTER, aspect=210 / 297)
assert abs(f_a4.shape[1] / f_a4.shape[0] - 210 / 297) < 0.01, "с aspect размер выхода должен держать пропорции W / H"
f_px, _ = rectify(scene, POSTER)
naive = f_px.shape[1] / f_px.shape[0]
print("TODO 2 ✔  · совпало %.1f %% · по A4: %d × %d (W/H %.3f) · «по сторонам в кадре»: %d × %d (W/H %.2f — врёт на %.0f %%)"
      % (match, f_a4.shape[1], f_a4.shape[0], f_a4.shape[1] / f_a4.shape[0], f_px.shape[1], f_px.shape[0], naive, 100 * (naive / (W / Hh) - 1)))
viz.grid({"size = (640, 905)": front, "aspect = A4": f_a4, "размер по сторонам в кадре": f_px}, cols=3, size=3.4)

### Точность кликов — измеряем, а не верим

«Четырёх точек достаточно» — достаточно для решения, не для точности (слайд 58). Добавим к четырём углам
шум клика σ, триста раз пересчитаем H и посмотрим, куда уезжают **узлы сетки плаката** на выпрямленном
виде. Второй опыт — тот же плакат, снятый втрое дальше (втрое меньше в кадре).

In [ ]:
x0, y0, cell = 60, 320, 130
grid_sheet = np.array([[x0 + i * cell, y0 + j * cell] for i in range(5) for j in range(5)], np.float64)   # 25 узлов сетки на листе
rng = np.random.default_rng(11)


def click_error(corners, sigma, n=300):
    """Шум клика σ на четырёх углах → медиана максимальной ошибки узла сетки на выпрямленном виде, px."""
    corners = np.float32(corners)
    grid_frame = apply_H(np.linalg.inv(cv2.getPerspectiveTransform(corners, dst)), grid_sheet)   # где узлы в кадре на самом деле
    errs = []
    for _ in range(n):
        Hn = cv2.getPerspectiveTransform(order_corners(corners + rng.normal(0, sigma, (4, 2))), dst)
        errs.append(np.linalg.norm(apply_H(Hn, grid_frame) - grid_sheet, axis=1).max())
    return float(np.median(errs))


small = ((POSTER - POSTER.mean(0)) / 3 + POSTER.mean(0)).astype(np.float32)          # тот же плакат втрое меньше в кадре
print("%-8s %22s %22s" % ("σ клика", "плакат 700 px в кадре", "плакат 230 px в кадре"))
errs = {s: (click_error(POSTER, s), click_error(small, s)) for s in (1, 2, 4)}
for s, (big, sm) in errs.items():
    print("%-8s %19.1f px %19.1f px" % (s, big, sm))
assert errs[1][0] < errs[2][0] < errs[4][0], "ошибка должна расти с σ"
assert 2.3 < errs[2][1] / errs[2][0] < 3.8, "втрое короче рычаг — примерно втрое больше ошибка"

### Своё фото: клик по четырём углам

Положите фото плаката, этикетки или документа под углом в `data/my-poster.jpg` (телефон, до ~2000 px по
длинной стороне). Три способа задать углы:

- **локально** — `RUN_CLICKER = True`: окно OpenCV, четыре клика в любом порядке, Enter — готово, `r` — заново;
- **Colab / без окна** — ячейка покажет фото с осями: впишите четыре угла в `MY_CORNERS` и перезапустите;
- **фото нет** — берётся плакат лекции.

`MY_ASPECT` — реальные пропорции объекта W / H (A4 — 210/297, визитка 90 × 50 — 1.8); `None` — «по сторонам
в кадре», и вы уже видели, на сколько это врёт.

In [ ]:
MY_PHOTO = "data/my-poster.jpg"       # своё фото плаката, этикетки, документа под углом
MY_CORNERS = None                     # например [[412, 233], [1580, 310], [1490, 1190], [380, 1020]] — в любом порядке
MY_ASPECT = 210 / 297                 # пропорции объекта W / H; None — по сторонам в кадре
RUN_CLICKER = False                   # True — окно OpenCV с кликами (только локально, не в Colab)


def pick_corners(img, title="4 klika po uglam, Enter - gotovo, r - zanovo"):
    """Окно OpenCV: четыре клика по углам → float32 (4, 2) в координатах исходного кадра (или None)."""
    pts, scale = [], min(1.0, 1200.0 / img.shape[1])                    # окно не шире 1200 px
    def on_click(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(pts) < 4:
            pts.append((x / scale, y / scale))
    cv2.namedWindow(title); cv2.setMouseCallback(title, on_click)
    while True:
        view = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        for p in pts:
            cv2.circle(view, (int(p[0] * scale), int(p[1] * scale)), 6, (0, 200, 255), -1)
        cv2.imshow(title, view)
        key = cv2.waitKey(30) & 0xFF
        if key == ord("r"):
            pts.clear()
        if key in (13, 10, ord("q")) or cv2.getWindowProperty(title, cv2.WND_PROP_VISIBLE) < 1:
            break
    cv2.destroyAllWindows()
    return np.float32(pts) if len(pts) == 4 else None


my = cio.imread(MY_PHOTO) if os.path.exists(MY_PHOTO) else None
if my is None:
    print("своего фото нет (%s) — берём плакат лекции; положите фото и перезапустите ячейку" % MY_PHOTO)
    my, MY_CORNERS = scene, (POSTER if MY_CORNERS is None else MY_CORNERS)
if RUN_CLICKER and not cvcourse.IN_COLAB:
    MY_CORNERS = pick_corners(my)
if MY_CORNERS is None:
    viz.show(my, "впишите четыре угла в MY_CORNERS по осям и перезапустите ячейку", size=9, axis=True)
else:
    mine, H_my = rectify(my, MY_CORNERS, aspect=MY_ASPECT)
    print("углы:", np.int32(order_corners(MY_CORNERS)).tolist(), "→ выпрямленный вид %d × %d" % (mine.shape[1], mine.shape[0]))
    viz.grid({"фото": my, "выпрямленный вид": mine}, cols=2, size=4.5)

### Bird's-eye view: та же матрица, другая плоскость

Слайд 54. Кадр с камеры робота: четыре точки **на асфальте** с известными координатами в метрах (в жизни —
конусы и рулетка; у нас сцена синтетическая, и точки посчитаны по модели камеры L2) → та же
`getPerspectiveTransform` → **вид сверху, 30 px/м**, где расстояния измеряются линейкой. Что платим:
плотность строк кадра на метр дороги падает как `f·h / Z²`, и дальние строки bird's-eye растягивает.

In [ ]:
CAM_FX, CAM_H, th = 1663.0, 1.2, np.deg2rad(12.0)                       # камера робота (модель L2): фокус, высота, наклон вниз
K = np.array([[CAM_FX, 0, 960], [0, CAM_FX, 540], [0, 0, 1]])
G2I = K @ np.array([[1, 0, 0], [0, -np.sin(th), CAM_H * np.cos(th)], [0, np.cos(th), CAM_H * np.sin(th)]])   # (X, Z, 1) дороги → кадр
ground = np.array([[-1.75, 5.0], [1.75, 5.0], [1.75, 15.0], [-1.75, 15.0]])    # четыре точки на асфальте, м: X вправо, Z вперёд
src_px = np.float32(apply_H(G2I, ground))                                       # они же в кадре — по ним кликают

PX_PER_M, X0, X1, Z0, Z1 = 30, -5.0, 5.0, 3.0, 23.0                            # вид сверху: 30 px/м, окно X −5…5 м, Z 3…23 м
to_bev = lambda XZ: np.float32([[(X - X0) * PX_PER_M, (Z1 - Z) * PX_PER_M] for X, Z in XZ])
H_bev = cv2.getPerspectiveTransform(src_px, to_bev(ground))
bev = cv2.warpPerspective(road, H_bev, (int((X1 - X0) * PX_PER_M), int((Z1 - Z0) * PX_PER_M)), flags=cv2.INTER_LINEAR)


def lane_width_m(Z):
    """Расстояние между линиями полосы (±1.75 м) на строке Z м вида сверху, в метрах."""
    row = int((Z1 - Z) * PX_PER_M)
    line = cv2.cvtColor(bev[row - 2:row + 3], cv2.COLOR_BGR2GRAY).mean(0)
    xs = np.where(line > 170)[0]; mid = bev.shape[1] / 2
    xs = xs[(np.abs(xs - mid) < 1.9 * PX_PER_M + 12) & (np.abs(xs - mid) > 10)]           # линии полосы, без края и осевой
    return float((xs[xs > mid].mean() - xs[xs < mid].mean()) / PX_PER_M)


rows_per_m = lambda Z: CAM_FX * CAM_H / Z ** 2                                  # строк кадра на метр дороги: dv/dZ ≈ f·h/Z²
horizon = apply_H(G2I, [[0.0, 1e9]])[0, 1]                                      # Z → ∞: строка горизонта
print("точки на асфальте (м) → кадр (px):", np.round(src_px).astype(int).tolist())
print("ширина полосы по виду сверху: на 8 м %.2f м, на 20 м %.2f м (истина 3.5)" % (lane_width_m(8), lane_width_m(20)))
print("строк кадра на метр дороги: на 5 м %.0f, на 20 м %.0f → на 20 м bird's-eye растягивает каждую строку кадра в %.0f раз"
      % (rows_per_m(5), rows_per_m(20), PX_PER_M / rows_per_m(20)))
print("горизонт — строка %.1f кадра: там w → 0, вид сверху «до горизонта» не существует" % horizon)
assert abs(lane_width_m(8) - 3.5) < 0.1 and abs(lane_width_m(20) - 3.5) < 0.15, "полоса 3.5 м должна измеряться на обоих расстояниях"
shown_road = road.copy(); cv2.polylines(shown_road, [np.int32(src_px)], True, viz.ORANGE, 3)
viz.grid({"кадр с камеры робота и четыре точки": shown_road, "bird's-eye view, 30 px/м (3…23 м)": bev}, cols=2, size=4.4)

### Вопрос, на который отвечаем вслух

Ширина полосы измерена верно и на 8, и на 20 метрах, но штрихи разметки вдали размыты. Что растянуто —
**данные или сетка**? На каком расстоянии на метр дороги приходится одна строка кадра (`f·h / Z²` = 1),
и можно ли там что-то измерить?

**Bird's-eye view своего стола (локально, с веб-камерой):** ячейка ниже с `RUN_BEV_CAMERA = True` — лист A4
на столе, камера смотрит под углом, четыре клика по углам листа → вид сверху **1 px = 1 мм** в реальном
времени. Положите рядом с листом телефон или линейку: его размер на картинке — в миллиметрах.

In [ ]:
RUN_BEV_CAMERA = False               # True: лист A4 на столе → 4 клика → вид сверху 1 px/мм с камеры, q — выход (только локально)
if RUN_BEV_CAMERA and not cvcourse.IN_COLAB:
    frames = cio.video_frames(0, max_frames=300)
    corners = pick_corners(next(frames), "4 klika po uglam lista A4, Enter")
    if corners is not None:
        pad = 150                                                                       # запас вокруг листа, мм
        sheet_mm = np.float32([[0, 0], [210, 0], [210, 297], [0, 297]]) + pad           # лист в мм
        H_desk = cv2.getPerspectiveTransform(order_corners(corners), sheet_mm)
        for f in frames:
            top = cv2.warpPerspective(f, H_desk, (210 + 2 * pad, 297 + 2 * pad))        # 1 px = 1 мм на плоскости стола
            cv2.rectangle(top, (pad, pad), (pad + 210, pad + 297), (0, 200, 255), 1)
            cv2.imshow("bird's-eye: 1 px = 1 mm (q - exit)", top)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        cv2.destroyAllWindows()
    frames.close()                                                                      # отпустить камеру

---

## TODO 3 — четыре интерполяции числом *(≈ 15 минут)*

Демо 2.3 и слайд 50. Честный эксперимент: **уменьшаем** картинку в `factor` раз с `INTER_AREA` — эталон
известен, это сама картинка, — и **увеличиваем обратно** каждым из четырёх флагов. Измеряем только
увеличение: PSNR к оригиналу и время.

Напишите `upscale_table(img, factor, flags)` → массив `float` формы `(len(flags), 2)`: столбец 0 — PSNR
в дБ, столбец 1 — время увеличения в мс (минимум из пяти прогонов). `flags` — имена флагов строками:
`("INTER_NEAREST", "INTER_LINEAR", "INTER_CUBIC", "INTER_LANCZOS4")`.

> **Подсказка.** `getattr(cv2, name)` — флаг по имени; `metrics.psnr(a, b)` из библиотеки курса;
> `1e3 * min(timeit.repeat(lambda: ..., number=1, repeat=5))` — миллисекунды. Уменьшать один раз, до цикла.

In [ ]:
FLAGS = ("INTER_NEAREST", "INTER_LINEAR", "INTER_CUBIC", "INTER_LANCZOS4")


def upscale_table(img, factor, flags=FLAGS):
    """Уменьшить в factor раз (INTER_AREA) и увеличить обратно каждым флагом → массив (len(flags), 2): PSNR к оригиналу, дБ; время, мс."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 7–9 строк: resize вниз с INTER_AREA один раз; цикл по флагам: getattr(cv2, name), resize вверх до (w, h), metrics.psnr, min(timeit.repeat(...))
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


src = cv2.resize(photo, (960, 640), interpolation=cv2.INTER_AREA)       # фото ИУ 960 × 640, как в демо
crop = src[230:350, 330:490]                                            # фасад 160 × 120 (демо 2.3)
t8 = np.asarray(upscale_table(crop, 8), np.float64)
assert t8.shape == (4, 2), "ожидался массив (флаги × [PSNR, мс])"
assert np.all(np.isfinite(t8)) and np.all(t8[:, 1] > 0), "время — положительное число миллисекунд"
ref = cv2.resize(cv2.resize(crop, (20, 15), interpolation=cv2.INTER_AREA), (160, 120), interpolation=cv2.INTER_CUBIC)
assert abs(t8[2, 0] - metrics.psnr(crop, ref)) < 1e-6, "PSNR в строке INTER_CUBIC должен совпасть с прямым вызовом (уменьшение — INTER_AREA)"
assert t8[0, 0] < t8[1, 0] < t8[2, 0] <= t8[3, 0] + 0.05, "на фото ожидался порядок NEAREST < LINEAR < CUBIC ≤ LANCZOS4"
assert 0.02 < t8[3, 1] < 100, "время — в миллисекундах: LANCZOS4 на 160 × 120 занимает десятые доли мс (сейчас %g)" % t8[3, 1]
if SRC != "синтетика":
    assert np.abs(t8[:, 0] - [19.9, 20.6, 21.8, 22.1]).max() < 0.15, "фасад ×8 должен дать числа слайда 50: 19.9 / 20.6 / 21.8 / 22.1 дБ"
print("TODO 3 ✔  · фасад 160 × 120, уменьшили ×8 и увеличили обратно:")
for name, (db, ms) in zip(FLAGS, t8):
    print("   %-15s %5.1f дБ   %6.3f мс" % (name, db, ms))

### Текст: где флаг уже не помогает

Та же таблица на странице текста при ×4 и ×8 — и четыре увеличенных фрагмента фасада глазами. Слайд 50
обещает: при ×8 на тексте все четыре флага дают 15 дБ, потому что штрих в два пикселя **уничтожен
уменьшением** — информации нет, и никакая интерполяция её не вернёт.

In [ ]:
for factor in (4, 8):
    t = np.asarray(upscale_table(text_page, factor))
    print("текст 640 × 480, ×%d:  " % factor + "  ".join("%s %.1f дБ (%.2f мс)" % (n.replace("INTER_", ""), db, ms) for n, (db, ms) in zip(FLAGS, t)))
    if factor == 8:
        assert t[:, 0].max() - t[:, 0].min() < 0.6, "при ×8 на тексте разница между флагами должна исчезнуть (< 0.6 дБ)"
tiny = cv2.resize(crop, (20, 15), interpolation=cv2.INTER_AREA)
ups = {n.replace("INTER_", "") + " %.1f дБ" % t8[i, 0]: cv2.resize(tiny, (160, 120), interpolation=getattr(cv2, n)) for i, n in enumerate(FLAGS)}
viz.grid({"оригинал 160 × 120": crop, **ups}, cols=5, size=2.6, suptitle="фасад: уменьшили ×8 и увеличили обратно")

### PSNR не видит геометрии

Слайд 59: сдвиньте картинку на один пиксель — глаз не заметит, PSNR обрушится. Поэтому в TODO 2 мерили
долю совпавших пикселей и ошибку узлов в пикселях, а не PSNR.

In [ ]:
g = cv2.cvtColor(photo, cv2.COLOR_BGR2GRAY)                              # фото 1920 × 1279 в сером, как на слайде
print("PSNR фото к самому себе, сдвинутому на 1 px: %.1f дБ; на 3 px: %.1f дБ — «артефакты», которых нет"
      % (metrics.psnr(g[:, 1:], g[:, :-1]), metrics.psnr(g[:, 3:], g[:, :-3])))

### Уменьшение: «AREA» в варпе не работает

Демо 2.4 и слайд 52 — самая коварная строка таблицы флагов, и ровно пункт «масштабирование с корректным
анти-алиасингом» из ДЗ 1. Уменьшаем кадр 1920 × 1279 в четыре раза `warpAffine`: эталон — Lanczos из
Pillow, как в L1. Флаг `INTER_AREA` в варпе **молча заменяется на `LINEAR`**; лечение — размыть до варпа.

In [ ]:
from PIL import Image
h_full = photo.shape[0]
size4 = (480, 320)                                                       # ×¼ от 1920 × 1279
ref4 = np.asarray(Image.fromarray(photo[..., ::-1]).resize(size4, Image.LANCZOS))[..., ::-1]   # чужой эталон
M4 = np.float32([[0.25, 0, -0.375], [0, 320 / h_full, (320 / h_full - 1) / 2]])                # масштаб ¼ с поправкой на полпикселя
lin4 = cv2.warpAffine(photo, M4, size4, flags=cv2.INTER_LINEAR)
area4 = cv2.warpAffine(photo, M4, size4, flags=cv2.INTER_AREA)          # молча LINEAR
blur4 = cv2.warpAffine(cv2.GaussianBlur(photo, (0, 0), 1.5), M4, size4, flags=cv2.INTER_LINEAR)
resize4 = cv2.resize(photo, size4, interpolation=cv2.INTER_AREA)
print("warpAffine LINEAR %.1f дБ · «AREA» %.1f дБ · разница между ними %d кодов" % (metrics.psnr(ref4, lin4), metrics.psnr(ref4, area4), np.abs(lin4.astype(int) - area4).max()))
print("GaussianBlur σ 1.5 → warpAffine LINEAR: %.1f дБ · cv2.resize INTER_AREA: %.1f дБ" % (metrics.psnr(ref4, blur4), metrics.psnr(ref4, resize4)))
assert np.array_equal(lin4, area4), "INTER_AREA в warpAffine должен совпасть с INTER_LINEAR до кода"
viz.grid({"warpAffine LINEAR": lin4[120:220, 160:320], "blur σ 1.5 → warpAffine": blur4[120:220, 160:320], "resize INTER_AREA": resize4[120:220, 160:320]}, cols=3, size=3.2)

### Вопрос, на который отвечаем вслух

Строка `INTER_CUBIC` выигрывает у `INTER_NEAREST` два децибела на фото и ноль — на тексте ×8. Что общего
у этих двух случаев с гистограммой из L3 («эквализация не создаёт информацию»)? И почему размытие перед
уменьшением — **не порча**, а лечение, хотя после него картинка «мыльнее»? (Строгий ответ — в L6.)

---

## ⭐ Звёздочка *(по желанию, +0.5, до конца следующей недели)*

**Свой билинейный `remap`.** Демо 2.2, слайд 65: семь строк NumPy — целая часть координаты даёт
левого верхнего соседа, дробная — веса, четыре соседа достаём индексацией массивом. Ячейка ниже сверяет его
с `cv2.remap` на аффинной карте (поворот 10°, масштаб 1.3) и с `warpPerspective` на карте гомографии плаката
(там уже есть деление на `w`).

Звёздочка — одно из двух: **(а)** замените ядро на бикубическое (Keys, a = −0.75; веса на слайде 49) и
сверьтесь с `INTER_CUBIC` по максимальному расхождению и PSNR; **(б)** постройте карту `remap` для
полярной развёртки (слайд 53) или для «бочки» `k₁ = −0.25` и пропустите через свой `bilinear` —
сдайте ноутбук с картинкой и числами.

In [ ]:
def bilinear(img, mx, my):
    """Свой remap с билинейной интерполяцией: dst[y, x] = img[my[y, x], mx[y, x]] по четырём соседям (граница — REPLICATE)."""
    h, w = img.shape[:2]
    x0 = np.clip(np.floor(mx).astype(int), 0, w - 2); y0 = np.clip(np.floor(my).astype(int), 0, h - 2)
    fx = np.clip(mx - x0, 0, 1)[..., None]; fy = np.clip(my - y0, 0, 1)[..., None]
    top = img[y0, x0] * (1 - fx) + img[y0, x0 + 1] * fx
    bot = img[y0 + 1, x0] * (1 - fx) + img[y0 + 1, x0 + 1] * fx
    return np.uint8(np.round(top * (1 - fy) + bot * fy))


ms = lambda fn: 1e3 * min(timeit.repeat(fn, number=1, repeat=5))
A = cv2.invertAffineTransform(cv2.getRotationMatrix2D((480, 320), 10, 1.3))            # обратная матрица 2 × 3: результат → источник
ys, xs = np.mgrid[:640, :960].astype(np.float32)
mx, my = [np.float32(a * xs + b * ys + c) for a, b, c in A]                            # карта dst → src, только float32
ref_a = cv2.remap(src, mx, my, cv2.INTER_LINEAR)
d_a = np.abs(bilinear(src, mx, my).astype(int) - ref_a)
print("аффинная карта против cv2.remap: макс %d, среднее %.2f кода · %.0f мс против %.1f мс у remap"
      % (d_a.max(), d_a.mean(), ms(lambda: bilinear(src, mx, my)), ms(lambda: cv2.remap(src, mx, my, cv2.INTER_LINEAR))))
assert d_a.max() <= 4, "OpenCV считает веса в фиксированной точке (5 бит): расхождение не больше 4 кодов"

Hinv = np.linalg.inv(H_ref)                                                             # лист → кадр
ys, xs = np.mgrid[:Hh, :W].astype(np.float64)
u, v, wgt = Hinv[0, 0] * xs + Hinv[0, 1] * ys + Hinv[0, 2], Hinv[1, 0] * xs + Hinv[1, 1] * ys + Hinv[1, 2], Hinv[2, 0] * xs + Hinv[2, 1] * ys + Hinv[2, 2]
px, py = np.float32(u / wgt), np.float32(v / wgt)                                       # деление на w — вот она, перспектива
mine_p = bilinear(scene, px, py)
d_p = np.abs(mine_p.astype(int) - front_ref)
print("карта гомографии против warpPerspective: макс %d, среднее %.2f кода, PSNR %.1f дБ" % (d_p.max(), d_p.mean(), metrics.psnr(front_ref, mine_p)))
viz.grid({"свой bilinear по карте H⁻¹": mine_p, "cv2.warpPerspective": front_ref}, cols=2, size=3.2)

---

## Мост к домашнему заданию

**ДЗ 1 «Фотолаборатория», дедлайн W6, клиника — занятие 5.** Пункт 3 «Геометрия» — это сегодняшние ячейки
почти без изменений:

| Сегодня | В ДЗ 1 |
|---------|--------|
| `order_corners` + `rectify` + `pick_corners` | пункт 3: **перспективное выпрямление по 4 кликам мышью** — на своём фото плаката или текста под углом |
| `aspect` в `rectify` и число «по сторонам в кадре врёт на 27 %» | обоснование размера выхода в отчёте: реальные пропорции, а не длины сторон |
| «AREA в варпе не работает»: размыть до варпа или `resize(INTER_AREA)` | пункт 3: **масштабирование с корректным анти-алиасингом** — с числом PSNR, как сегодня |
| ячейка ниже — поворот с сохранением кадра (демо 1.1) | пункт 3: **поворот с сохранением всего кадра** |
| `upscale_table`: таблица «флаг → PSNR, мс» | тот же приём для любого параметра: не «стало красиво», а число |

Дальше те же функции пойдут в ДЗ 2 «Сканер документов» (W6–W9): там четыре угла листа найдёт не клик,
а `Canny` + контуры / `HoughLinesP` + RANSAC — L7. Полное ТЗ — `homeworks/hw1-photolab/README.md`.

In [ ]:
h, w = photo.shape[:2]
M = cv2.getRotationMatrix2D((w / 2, h / 2), 30, 1.0)                   # 2 × 3: T(c)·R·T(−c) — поворот вокруг центра
cos, sin = abs(M[0, 0]), abs(M[0, 1])
nw, nh = round(w * cos + h * sin), round(w * sin + h * cos)            # холст по повёрнутым углам
M[0, 2] += nw / 2 - w / 2                                              # сдвинуть так, чтобы центр кадра попал в центр холста
M[1, 2] += nh / 2 - h / 2
full_rot = cv2.warpAffine(photo, M, (nw, nh), borderValue=(255, 255, 255))
cropped = cv2.warpAffine(photo, cv2.getRotationMatrix2D((w / 2, h / 2), 30, 1.0), (w, h))   # прежний dsize — углы отрезаны
kept = cv2.warpAffine(np.ones((h, w), np.uint8), cv2.getRotationMatrix2D((w / 2, h / 2), 30, 1.0), (w, h)).sum() / (h * w)
print("кадр %d × %d → холст %d × %d; с прежним dsize теряется %.0f %% пикселей" % (w, h, nw, nh, 100 * (1 - kept)))
viz.grid({"прежний dsize: углы отрезаны": cropped, "холст по повёрнутым углам": full_rot}, cols=2, size=4)

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–3 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, **почему варп исполняется обратно** (для каждого пикселя результата ищем
      источник, а не наоборот) и что делает деление на `w`;
- [ ] на вопрос «почему размер выхода нельзя брать по сторонам в кадре» и «что растянуто вдали
      в bird's-eye view» есть внятный ответ.

**2 % итоговой оценки**, ставится в классе. Досылать ничего не нужно.